In [10]:
import numpy as np
import pandas as pd


class ConformalPrediction:
    """Conformal prediction with marginal coverage guarantee.

    Parameters
    ----------
    alpha : float
        Target miscoverage rate.  A prediction set will contain the true
        label with probability >= 1 - alpha.
    """

    def __init__(self, alpha: float = 0.1):
        if not 0 < alpha < 1:
            raise ValueError("alpha must be in (0, 1)")
        self.alpha = alpha
        self.qhat = None

    def calibrate(self, logits: np.ndarray, labels: np.ndarray) -> None:
        """Compute the conformal quantile from calibration data."""
        n = logits.shape[0]
        scores = 1.0 - logits[np.arange(n), labels]
        q_level = np.ceil((n + 1) * (1 - self.alpha)) / n
        self.qhat = np.quantile(scores, min(q_level, 1.0), method="higher")

    def predict(self, logits: np.ndarray) -> np.ndarray:
        """Return binary prediction-set matrix (N x C)."""
        if self.qhat is None:
            raise RuntimeError("Model not calibrated – call .calibrate() first")
        return logits >= (1.0 - self.qhat)

    def __call__(self, logits: np.ndarray) -> np.ndarray:
        return self.predict(logits)


def evaluate(crc, test_logits, test_labels):
    """Return empirical coverage and average prediction-set size."""
    pred_sets = crc(test_logits)
    coverage = pred_sets[np.arange(pred_sets.shape[0]), test_labels].mean()
    avg_size = pred_sets.sum(axis=1).mean()
    return coverage, avg_size


df = pd.read_parquet("assets/slide_predictions_test_split0_conch.parquet.gzip")
prob_cols = [c for c in df.columns if c.startswith("prob_")]
logits = df[prob_cols].values.astype(np.float64)          # shape (N, 46)
labels = df["Outcome y_true"].values.astype(np.int64)     # shape (N,)

seed = 42
permutation_size = 100
calibration_size = 500

results = {
    alpha: {"coverage": [], "size": []} for alpha in (0.05, 0.01)
}

for s in range(seed, seed + permutation_size):
    rng = np.random.RandomState(s)
    idx = rng.permutation(len(logits))
    cal_logits, cal_labels = logits[idx[:calibration_size]], labels[idx[:calibration_size]]
    test_logits, test_labels = logits[idx[calibration_size:]], labels[idx[calibration_size:]]

    for alpha in (0.05, 0.01):
        crc = ConformalPrediction(alpha=alpha)
        crc.calibrate(cal_logits, cal_labels)
        cov, size = evaluate(crc, test_logits, test_labels)
        results[alpha]["coverage"].append(cov)
        results[alpha]["size"].append(size)

for alpha, label in [(0.05, "α = 0.05"), (0.01, "α = 0.01")]:
    covs = results[alpha]["coverage"]
    sizes = results[alpha]["size"]
    print(
        f"{label:>10s}  "
        f"coverage = {np.mean(covs):.4f} ± {np.std(covs):.4f}  |  "
        f"avg set size = {np.mean(sizes):.2f} ± {np.std(sizes):.2f}"
    )

  α = 0.05  coverage = 0.9515 ± 0.0111  |  avg set size = 2.73 ± 0.23
  α = 0.01  coverage = 0.9921 ± 0.0045  |  avg set size = 8.16 ± 1.82
